# Predicting code-switches and their duration (XLM-R vs. Qwen3)

**Data:** SwitchLingua_text (`first_language` x `second_language`), the pairs listed in `config.PAIRS` (enable needed languages there). All active pairs are trained together in one model; evaluation is reported per pair.

**Task:** one shared encoder with two heads
- **switch head** (binary): will the next unit change language?
- **duration head** (`N_DUR = 4` classes): the length class of the upcoming segment.
  - `0` = no switch
  - `1` = the new-language segment lasts 1-2 units
  - `2` = 3-7 units
  - `3` = >= 8 units

  Class `0` is trained with the rest, so the head also sees no-switch positions. At evaluation and inference the duration is read only from classes `1..3` (`argmax` over those logits), and `dur_acc` / `dur_MAE` are computed only where the gold label is a real switch (`dur > 0`). The ordinal L1 penalty is applied only to switch positions and only over classes `1..3`, because class `0` is not "shorter" than class `1`.

**LID labels** (0 = non-English L1, 1 = English) are only used to *generate* the labels (kept as `last_lang` / `next_lang` for inspection); they are never model inputs. Script decides LID (Han / Devanagari / ... -> 0, Latin -> 1); Latin-script pairs (and **(TO-DO)** `hi-en`, which also covers romanized Hindi) use wordfreq dictionaries per word, with ambiguous words resolved from the nearest determined neighbours.

**Units:** Han characters and kana = one character; all other scripts = one word. Context = everything from the start of the sentence to the current position, identically in training and inference.

> Duration bins count units, which are not fully comparable across pairs (characters vs. words), so always read results per pair. `dur_MAE` is measured in bins (classes 1-3, so each error is at most 2), not in units; prefer `dur_acc` when comparing.

## Project layout
```
pipeline.ipynb          <- this notebook: pipeline, results, commentary
codeswitch/
  config.py             hyperparameters, language pairs, paths
  utils.py              device, seeding, memory monitoring, class weights
  lid.py                unit segmentation + language identification
  data.py               corpus loading, split by conversation, example generation
  metrics.py            score / per-pair score / result tables
  baselines.py          B0 prior, B1 lookup table, B2 char n-gram LR
  trainer.py            shared training loop (differential LR, optional best-epoch restore)
  xlmr.py               XLM-R dataset, model, evaluate, predict
  qwen.py               Qwen3 dataset, model, evaluate, predict
```
Sections 4-5 are the XLM-R pipeline; section 6 is Qwen3. Both use the same train/val sentences, the same head definitions and the same `score` function, so results are directly comparable. Change hyperparameters in `codeswitch/config.py` (picked up by `%autoreload`).

## Metrics

Computed after every epoch on the validation set, pooled (`ALL`) and per pair. Both models use the same code (`metrics.py`), so the numbers are comparable.

### Primary (best-epoch selection)
Selection score = `0.5 * (switch_PR_AUC + dur_macro_F1)` on `ALL` (`trainer.select_score`).

| Metric | Meaning |
|---|---|
| `switch_PR_AUC` | Area under the precision-recall curve of the switch head, computed from `P(switch)`. Threshold-free and robust to the ~10% positive rate. The random baseline equals the switch share. |
| `dur_macro_F1` | Macro-averaged F1 over duration classes 1-3, **only at positions whose gold label is a real switch** (`dur > 0`); the prediction is the `argmax` over classes 1-3. Each class counts equally, so the rare long segments matter as much as the common short ones. |

### Auxiliary
| Metric | Meaning |
|---|---|
| `switch_P`, `switch_R`, `switch_F1` | Precision / recall / F1 of the switch head at the argmax (0.5) threshold. |
| `dur_R1`, `dur_R2`, `dur_R3` | Per-class recall of the duration head (1-2 / 3-7 / >=8 units), switch positions only. Shows which class the head neglects. |
| `dur_acc`, `dur_MAE` | Duration accuracy and mean absolute error in bins (max 2), switch positions only. |
| `joint_acc` | Over **all** positions: gold switch -> counts as correct only if a switch is predicted **and** the duration class is right; gold no-switch -> correct if no switch is predicted. |
| `val_ce_sw`, `val_ce_dur` | Plain cross-entropy (`-log p(gold class)`), no class weights, no `LAMBDA_DUR`, no label smoothing, no ordinal term. `val_ce_sw` is over all positions; `val_ce_dur` only at gold switches and over classes 1-3. Use these to **compare models and heads**. |

### Losses for the train / validation curves (`ALL` row only)
| Metric | Meaning |
|---|---|
| train `loss (sw / dur)` | Training objective, averaged over the epoch (dropout on). |
| `val_loss`, `val_loss_sw`, `val_loss_dur` | Validation loss with **exactly the training formula** (class weights, `LAMBDA_DUR`, Qwen's label smoothing and ordinal term). Compare with the train line to detect overfitting: train keeps falling while val rises. Not comparable across XLM-R and Qwen3, since their duration losses differ (XLM-R: all positions; Qwen3: smoothing + ordinal). |

> `switch_PR_AUC` is the only switch metric that does not depend on the 0.5 threshold; read `switch_P` / `switch_R` together with it.

## 0. Setup

In [1]:
# !pip install -q ipykernel transformers torch numpy pandas scikit-learn wordfreq psutil
%load_ext autoreload
%autoreload 2

import numpy as np
from codeswitch import config as C
from codeswitch.utils import DEVICE, set_seed, mem, free
from codeswitch import lid, data, baselines, xlmr, qwen
from codeswitch.metrics import results_table, save_results, print_scores

set_seed(C.SEED)
print("device:", DEVICE)
results = {}      # {model name: {pair or "ALL": metrics}}

/Users/WanderingCorn/Documents/Northeastern/CS6140/codeswitch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Segmentation + LID self-test
One example sentence per pair, printed as `(unit, label)`; `0` = L1, `1` = English.

In [2]:
lid.selftest()

zh-en [('我', 0), ('喜', 0), ('欢', 0), ('this', 1), ('movie', 1), ('因', 0), ('为', 0), ("it's", 1), ('很', 0), ('好', 0), ('看', 0)]
yue-en [('佢', 0), ('今', 0), ('日', 0), ('meeting', 1), ('完', 0), ('咗', 0), ('先', 0), ('返', 0), ('屋', 0), ('企', 0)]
hi-en [('मुझे', 0), ('यह', 0), ('movie', 1), ('बहुत', 0), ('पसंद', 0), ('आई', 0), ('because', 1), ('it', 1), ('was', 1), ('amazing', 1)]
hi-en [('mujhe', 1), ('yeh', 1), ('movie', 1), ('bahut', 1), ('pasand', 1), ('aayi', 1), ('because', 1), ('it', 1), ('was', 1), ('amazing', 1)]
es-en [('Ayer', 0), ('fui', 0), ('al', 0), ('cine', 0), ('con', 0), ('mi', 0), ('amigo', 0), ('and', 1), ('we', 1), ('watched', 1), ('a', 1), ('really', 1), ('good', 1), ('movie', 1), ('pero', 0), ('no', 0), ('me', 0), ('gustó', 0), ('el', 0), ('final', 0)]
ar-en [('أنا', 0), ('أحب', 0), ('هذا', 0), ('movie', 1), ('كثيراً', 0), ('because', 1), ("it's", 1), ('great', 1)]
ja-en [('今', 0), ('日', 0), ('は', 0), ('meeting', 1), ('が', 0), ('あ', 0), ('る', 0), ('の', 0), ('で', 0), ("

## 2. Load the corpus
Each CSV row is a conversation; `data_generation_result` is a stringified list of sentences. Sentences are expanded to records tagged with `pair` and `conv` so the split can be done per conversation. The health check flags pairs whose L1 share is < 5% (LID not applicable, e.g. romanized text).

In [3]:
records = data.load_records()
data.corpus_report(records)

first_language  second_language
Cantonese       English            119248
Arabic          English              8548
Philippines     English              8537
German          French               8511
Chinese         English              8458
Vietnamese      English              8457
Malay           English              8386
Japanese        English              8385
Hindi           English              8381
Korean          English              8266
Spanish         English              8110
French          English              8070
Russian         English              8017
Italian         English              7979
German          English              6819

36 sentences; 0 conversations failed to parse -> parse_failed_conversations.txt
yue-en  sentences     36 | L1 units      625 | EN units      231 | sentences with a switch 91.7%
yue-en | 聽講DeepSeek的AI模型真係好犀利，竟然可以同美國科技公司嘅產品媲美。I think it's a game changer for the tech industry.
yue-en | 呢個模型嘅成功令全球市場都震動，令我諗起全球人才競爭真係好激烈。 It's fascinating how

## 3. Split dataset split and build example
**Dataset split:** Training and validation dataset are split by conversations. Sentences from the same conversation cannot be assigned to differentdatasets.

**Example:** Every stopping point (after a unit, or after each punctuation mark / number) becomes one example: `text` = prefix, `switch` = does the next unit change language, `duration` = length class of the new segment (0 if no switch).

**Variable naming:** rec - conversation, raw - prefix text,

In [4]:
train_recs, val_recs = data.split_records(records)
train_raw = data.build_examples(train_recs)
val_raw = data.build_examples(val_recs)

for e in data.make_examples("मुझे यह movie बहुत पसंद आई, because it's amazing।", "hi-en")[:8]:
    print(e)
print()
data.example_report(train_raw, val_raw)

{'text': 'मुझे', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह', 'pair': 'hi-en', 'switch': 1, 'duration': 1, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'en', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत पसंद', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'L1', 'next_lang': 'L1'}
{'text': 'मुझे यह movie बहुत पसंद आई', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie बहुत पसंद आई,', 'pair': 'hi-en', 'switch': 1, 'duration': 2, 'last_lang': 'L1', 'next_lang': 'en'}
{'text': 'मुझे यह movie बहुत पसंद आई, because', 'pair': 'hi-en', 'switch': 0, 'duration': 0, 'last_lang': 'en', 'next_lang': 'en'}

train: 773 examples, switch=1 share 11.0%
duration class counts: [688, 26, 30,

## 4. Naive baselines (each pair trained and evaluated separately)

- **B0 prior:** always / never predict a switch; duration = most common class (lower bound)
- **B1 lookup:** `P(switch | current language, run length, trailing punctuation)`; duration = mode per language. No text representation at all
- **B2 char n-gram + LR:** TF-IDF on the last 6 characters. Surface form, no pretraining

If XLM-R cannot beat B1, the signal is essentially run length + punctuation; if it cannot beat B2, pretraining adds nothing.

In [5]:
results.update(baselines.run_baselines(train_raw, val_raw))

for metric in ["switch_PR_AUC", "switch_F1", "switch_P", "switch_R", "dur_macro_F1"]:   # B0 has no probabilities -> PR-AUC is NaN
    print(f"\n== {metric} ==")
    display(results_table(results, metric))


== switch_PR_AUC ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
yue-en,NaN,NaN,0.336,0.485



== switch_F1 ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
yue-en,0.147,0.0,0.345,0.588



== switch_P ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
yue-en,0.079,0.0,0.263,0.714



== switch_R ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
yue-en,1.0,0.0,0.5,0.5



== dur_macro_F1 ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,NaN,NaN,NaN,NaN
yue-en,0.0,0.0,0.0,0.222


## 5. XLM-R (shared encoder + two heads on the `<mask>` position)

Training uses **length-bucketed batches** to reduce padding. Negative downsampling is available (`KEEP_NEG`) but **disabled** (`KEEP_NEG = 1.0`): all examples are used and the imbalance is handled by the square-root class weights, as in Qwen3. Validation is never downsampled.

In [6]:
free(); mem("before XLM-R")
xlmr_tok, xlmr_model, xlmr_train_dl, xlmr_val_dl = xlmr.build(train_raw, val_raw)
mem("XLM-R loaded")

[mem before XLM-R] MPS alloc 0.0 GB | driver 0.0 GB | RSS 3.1 GB
downsampling: 773 -> 773, switch=1 share 11.0%
switch class weights: [0.7495153546333313, 2.132383346557617]


W1112 12:34:35.905000 10000 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7770.92it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[mem XLM-R loaded] MPS alloc 1.1 GB | driver 1.1 GB | RSS 2.4 GB


In [7]:
history = xlmr.train(xlmr_model, xlmr_train_dl, xlmr_val_dl)
results["XLM-R"] = xlmr.evaluate(xlmr_model, xlmr_val_dl)

[mem XLM-R step 0] MPS alloc 3.4 GB | driver 6.4 GB | RSS 1.4 GB
epoch 1: train loss 1.6652 (sw 0.5964 | dur 1.0689)
  val [ALL   ] {'switch_PR_AUC': 0.561, 'dur_macro_F1': 0.0, 'switch_P': 0.0, 'switch_R': 0.0, 'switch_F1': 0.0, 'dur_R1': 0.0, 'dur_R2': 0.0, 'dur_R3': 0.0, 'joint_acc': 0.921, 'dur_acc': 0.0, 'dur_MAE': 1.3, 'val_ce_sw': 0.2483, 'val_ce_dur': 1.5233, 'val_loss': np.float64(0.9663), 'val_loss_sw': np.float64(0.5288), 'val_loss_dur': np.float64(0.4376)}
  val [yue-en] {'switch_PR_AUC': 0.561, 'dur_macro_F1': 0.0, 'switch_P': 0.0, 'switch_R': 0.0, 'switch_F1': 0.0, 'dur_R1': 0.0, 'dur_R2': 0.0, 'dur_R3': 0.0, 'joint_acc': 0.921, 'dur_acc': 0.0, 'dur_MAE': 1.3, 'val_ce_sw': 0.2483, 'val_ce_dur': 1.5233}
  ^ best so far
[mem XLM-R epoch 1] MPS alloc 3.4 GB | driver 3.5 GB | RSS 3.6 GB
epoch 2: train loss 0.9327 (sw 0.4757 | dur 0.4569)
  val [ALL   ] {'switch_PR_AUC': 0.704, 'dur_macro_F1': 0.061, 'switch_P': 1.0, 'switch_R': 0.5, 'switch_F1': 0.667, 'dur_R1': 0.0, 'dur_R2'

### XLM-R inference on partially typed text

In [8]:
probes = ["我喜欢", "我喜欢 this", "I think 这个", "今天 weather 不错，",
          "मुझे यह", "मुझे यह movie", "佢今日 meeting", "聽講DeepSeek的AI"]
for s in probes:
    print(f"{s!r:25} {xlmr.predict(xlmr_model, xlmr_tok, s)}")

'我喜欢'                     {'p_switch': 0.017, 'duration_if_switch': 3, 'p_duration': [0.995, 0.001, 0.001, 0.002]}
'我喜欢 this'                {'p_switch': 0.09, 'duration_if_switch': 3, 'p_duration': [0.983, 0.006, 0.004, 0.007]}
'I think 这个'              {'p_switch': 0.01, 'duration_if_switch': 3, 'p_duration': [0.997, 0.001, 0.001, 0.001]}
'今天 weather 不错，'          {'p_switch': 0.413, 'duration_if_switch': 3, 'p_duration': [0.849, 0.034, 0.057, 0.059]}
'मुझे यह'                 {'p_switch': 0.004, 'duration_if_switch': 1, 'p_duration': [0.999, 0.0, 0.0, 0.0]}
'मुझे यह movie'           {'p_switch': 0.002, 'duration_if_switch': 1, 'p_duration': [0.999, 0.001, 0.0, 0.0]}
'佢今日 meeting'             {'p_switch': 0.002, 'duration_if_switch': 1, 'p_duration': [0.997, 0.001, 0.001, 0.001]}
'聽講DeepSeek的AI'           {'p_switch': 0.95, 'duration_if_switch': 2, 'p_duration': [0.188, 0.253, 0.306, 0.252]}


In [9]:
# Free memory before loading Qwen
del xlmr_model, xlmr_train_dl, xlmr_val_dl
free(); mem("after XLM-R cleanup")

[mem after XLM-R cleanup] MPS alloc 0.0 GB | driver 0.0 GB | RSS 1.9 GB


## 6. Qwen3 (decoder-only)

Differences from the XLM-R version:
- **One sentence per sample:** the causal mask lets one forward pass produce predictions for all stopping points
- **Labels on the last token of each chunk** (a decoder only knows what a unit is after reading all of it); other positions are `-100`
- **Chunked tokenization:** the sentence is split at stopping points and each chunk tokenized separately, so stopping points always fall on token boundaries
- **Heads on every position's hidden state;** the loss ignores `-100`
- Embedding is frozen (memory + overfitting); duration loss adds label smoothing and an ordinal L1 term
- Labelled positions match the XLM-R examples one-to-one (asserted in `check_alignment`)

In [10]:
free(); mem("before Qwen")
qwen_tok, qwen_model, qwen_train_dl, qwen_val_dl, qwen_train_ds = qwen.build(train_recs, val_recs)
qwen.show_alignment(train_recs, qwen_tok)
mem("Qwen loaded")

[mem before Qwen] MPS alloc 0.0 GB | driver 0.0 GB | RSS 1.9 GB
0 / 31 training sentences exceed QWEN_MAX_LEN=256 tokens (tail labels dropped)
switch class weights: [0.7495153546333313, 2.132383346557617]


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6229.38it/s]


yue-en | 聽講DeepSeek的AI模型真係好犀利，竟然可以同美國科技公司嘅產品媲美。I think it's a game changer for the tech industry.
'èģ½'          switch=   0 dur=   0
'è¬Ľ'          switch=   1 dur=   1
'Deep'         switch=-100 dur=-100
'Seek'         switch=   1 dur=   1
'çļĦ'          switch=   1 dur=   1
'AI'           switch=   1 dur=   3
'æ¨¡'          switch=   0 dur=   0
'åŀĭ'          switch=   0 dur=   0
'çľŁ'          switch=   0 dur=   0
'ä¿Ĥ'          switch=   0 dur=   0
'å¥½'          switch=   0 dur=   0
'çĬĢ'          switch=   0 dur=   0
'åĪ©'          switch=   0 dur=   0
'ï¼Į'          switch=   0 dur=   0
'ç«Ł'          switch=   0 dur=   0
'çĦ¶'          switch=   0 dur=   0
'åı¯'          switch=   0 dur=   0
'ä»¥'          switch=   0 dur=   0
'åĲĮ'          switch=   0 dur=   0
'ç¾İ'          switch=   0 dur=   0
'åľĭ'          switch=   0 dur=   0
'ç§ĳ'          switch=   0 dur=   0
'æĬĢ'          switch=   0 dur=   0
'åħ¬'          switch=   0 dur=   0
'åı¸'          switch=   0 dur=   0
'å

In [11]:
history_q = qwen.train(qwen_model, qwen_train_dl, qwen_val_dl)   # restores the best epoch
results["Qwen3"] = qwen.evaluate(qwen_model, qwen_val_dl)

[mem Qwen step 0] MPS alloc 5.9 GB | driver 9.8 GB | RSS 2.2 GB
epoch 1: train loss 4.3168 (sw 1.4012 | dur 2.9157)
  val [ALL   ] {'switch_PR_AUC': 0.126, 'dur_macro_F1': 0.19, 'switch_P': 0.113, 'switch_R': 0.7, 'switch_F1': 0.194, 'dur_R1': 0.0, 'dur_R2': 0.0, 'dur_R3': 0.4, 'joint_acc': 0.492, 'dur_acc': 0.4, 'dur_MAE': 0.8, 'val_ce_sw': 0.7792, 'val_ce_dur': 1.4107, 'val_loss': np.float64(2.2604), 'val_loss_sw': np.float64(0.75), 'val_loss_dur': np.float64(1.5104)}
  val [yue-en] {'switch_PR_AUC': 0.126, 'dur_macro_F1': 0.19, 'switch_P': 0.113, 'switch_R': 0.7, 'switch_F1': 0.194, 'dur_R1': 0.0, 'dur_R2': 0.0, 'dur_R3': 0.4, 'joint_acc': 0.492, 'dur_acc': 0.4, 'dur_MAE': 0.8, 'val_ce_sw': 0.7792, 'val_ce_dur': 1.4107}
  ^ best so far
[mem Qwen epoch 1] MPS alloc 5.9 GB | driver 8.8 GB | RSS 2.0 GB
epoch 2: train loss 2.1119 (sw 0.7376 | dur 1.3743)
  val [ALL   ] {'switch_PR_AUC': 0.15, 'dur_macro_F1': 0.25, 'switch_P': 0.13, 'switch_R': 0.3, 'switch_F1': 0.182, 'dur_R1': 0.0, 'du

### Qwen3 inference on partially typed text

In [12]:
for s in probes:
    print(f"{s!r:25} Qwen3: {qwen.predict(qwen_model, qwen_tok, s)}")

'我喜欢'                     Qwen3: {'p_switch': 0.228, 'duration_if_switch': 3, 'p_duration': [0.843, 0.056, 0.025, 0.076]}
'我喜欢 this'                Qwen3: {'p_switch': 0.136, 'duration_if_switch': 1, 'p_duration': [0.668, 0.23, 0.021, 0.081]}
'I think 这个'              Qwen3: {'p_switch': 0.771, 'duration_if_switch': 1, 'p_duration': [0.885, 0.066, 0.024, 0.026]}
'今天 weather 不错，'          Qwen3: {'p_switch': 0.504, 'duration_if_switch': 2, 'p_duration': [0.881, 0.039, 0.059, 0.021]}
'मुझे यह'                 Qwen3: {'p_switch': 0.575, 'duration_if_switch': 1, 'p_duration': [0.353, 0.545, 0.01, 0.092]}
'मुझे यह movie'           Qwen3: {'p_switch': 0.065, 'duration_if_switch': 1, 'p_duration': [0.369, 0.602, 0.001, 0.028]}
'佢今日 meeting'             Qwen3: {'p_switch': 0.141, 'duration_if_switch': 1, 'p_duration': [0.448, 0.499, 0.037, 0.016]}
'聽講DeepSeek的AI'           Qwen3: {'p_switch': 0.585, 'duration_if_switch': 3, 'p_duration': [0.589, 0.066, 0.023, 0.322]}


## 7. Comparison
Rows = language pairs (`ALL` = pooled, only for the jointly trained models); baselines are trained per pair.

In [13]:
for metric in ["switch_PR_AUC", "dur_macro_F1", "switch_P", "switch_R",
               "dur_R1", "dur_R2", "dur_R3", "joint_acc", "val_ce_sw", "val_ce_dur"]:
    print(f"\n== {metric} ==")
    display(results_table(results, metric))
save_results(results, "results.json")


== switch_PR_AUC ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,0.766,0.209
yue-en,NaN,NaN,0.336,0.485,0.766,0.209



== dur_macro_F1 ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,0.154,0.25
yue-en,0.0,0.0,0.0,0.222,0.154,0.25



== switch_P ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,0.583,0.25
yue-en,0.079,0.0,0.263,0.714,0.583,0.25


... (10 outputs left out)


== val_ce_sw ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,0.2073,0.3466
yue-en,NaN,NaN,NaN,NaN,0.2073,0.3466



== val_ce_dur ==


,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR,XLM-R,Qwen3
ALL,NaN,NaN,NaN,NaN,1.0342,0.7345
yue-en,NaN,NaN,NaN,NaN,1.0342,0.7345
